# Loan Default Prediction — Corrected Pipeline (v3)
Fixes / additions in this version:
1. Economic indicators merge (proper country/year matching, forward-fill 2024→2023)
2. Customer-level leakage fix (GroupShuffleSplit / GroupKFold by `customer_id`, not row-level split)
3. Feature engineering (repayment ratio, funding ratio, prior-loan count per customer)
4. CatBoost trained alongside LightGBM on a quick single split, best algorithm picked by validation AUC
5. **NEW:** 5-fold GroupKFold cross-validation using the winning algorithm — out-of-fold predictions
   give a more robust F1 threshold, and test predictions are averaged across all 5 folds
   (usually a real accuracy bump over a single train/val split, and less luck-dependent)
6. F1-optimized threshold for binary submission (metric = F1 Score, target = 0/1)

**Before running:** edit `DATA_DIR` and filenames in the Config cell to match where your CSVs live.
Requires: `pandas numpy scikit-learn lightgbm catboost` (`pip install catboost` if missing).

## 0. Config — edit these paths

In [ ]:
import os

DATA_DIR = "../data"  # put Train, Test and economic_indicators CSVs here (not committed to git)

TRAIN_FILE = "Train (1).csv"
TEST_FILE = "Test.csv"
ECON_FILE = "economic_indicators.csv"

TRAIN_PATH = os.path.join(DATA_DIR, TRAIN_FILE)
TEST_PATH = os.path.join(DATA_DIR, TEST_FILE)
ECON_PATH = os.path.join(DATA_DIR, ECON_FILE)

for p in [TRAIN_PATH, TEST_PATH, ECON_PATH]:
    print(p, "->", "FOUND" if os.path.exists(p) else "MISSING -- fix the path above")

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import lightgbm as lgb
from catboost import CatBoostClassifier

pd.set_option("display.max_columns", 100)

## 2. Load raw data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
economic_df = pd.read_csv(ECON_PATH)

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("economic_df:", economic_df.shape)

print("\nTrain countries:", train_df['country_id'].unique())
print("Test countries:", test_df['country_id'].unique())
print("Econ countries:", economic_df['Country'].unique())

## 3. Target distribution (imbalance check)

In [ ]:
print(train_df['target'].value_counts(normalize=True))

## 4. Clean & pivot economic indicators

Key fixes vs. the earlier broken attempt:
- Filter to only Ghana + Kenya (avoid noise from other countries in the file)
- Keep `Country` as a plain string (do NOT one-hot encode before merging)
- Loan data goes up to 2024, but economic data only goes to 2023 — forward-fill 2023 values into 2024 per country so nothing is left NaN just because of the year gap

In [ ]:
indicator_rename = {
    'Inflation, consumer prices (annual %)': 'Inflation_pct',
    'Official exchange rate (LCU per US$, period average)': 'Exchange_rate',
    'Real interest rate (%)': 'Real_interest_rate_pct',
    'Average precipitation in depth (mm per year)': 'Precipitation_mm',
    'Deposit interest rate (%)': 'Deposit_interest_rate_pct',
    'Lending interest rate (%)': 'Lending_interest_rate_pct',
    'Interest rate spread (lending rate minus deposit rate, %)': 'Interest_rate_spread_pct',
    'Fossil fuel energy consumption (% of total)': 'Fossil_fuel_pct',
    'Unemployment rate': 'Unemployment_pct',
}

econ = economic_df.copy()
econ['Indicator'] = econ['Indicator'].map(indicator_rename).fillna(econ['Indicator'])

relevant_countries = ['Ghana', 'Kenya']
econ = econ[econ['Country'].isin(relevant_countries)].copy()
print("Filtered econ shape:", econ.shape)

year_cols = [c for c in econ.columns if c.startswith('YR')]
econ_melted = econ.melt(id_vars=['Country', 'Indicator'], value_vars=year_cols,
                         var_name='Year', value_name='Value')
econ_melted['Year'] = econ_melted['Year'].str.replace('YR', '').astype(int)

econ_pivot = econ_melted.pivot_table(index=['Country', 'Year'], columns='Indicator',
                                      values='Value').reset_index()
econ_pivot = econ_pivot.rename(columns={'Country': 'country_id'})

print("Pivoted econ shape:", econ_pivot.shape)
print("Econ year range:", econ_pivot['Year'].min(), "-", econ_pivot['Year'].max())
econ_pivot.head()

In [ ]:
# Forward-fill missing future years (e.g. 2024) using each country's most recent known year
max_econ_year = econ_pivot['Year'].max()

extra_rows = []
for country in relevant_countries:
    latest = econ_pivot[econ_pivot['country_id'] == country].sort_values('Year').iloc[-1].copy()
    for future_year in range(max_econ_year + 1, 2026):
        new_row = latest.copy()
        new_row['Year'] = future_year
        extra_rows.append(new_row)

if extra_rows:
    econ_pivot = pd.concat([econ_pivot, pd.DataFrame(extra_rows)], ignore_index=True)

print("Econ pivot shape after extending years:", econ_pivot.shape)
print("Econ year range now:", econ_pivot['Year'].min(), "-", econ_pivot['Year'].max())

## 5. Prepare loan data: extract Year, then merge economic data

In [ ]:
train_df['disbursement_date'] = pd.to_datetime(train_df['disbursement_date'])
train_df['Year'] = train_df['disbursement_date'].dt.year

test_df['disbursement_date'] = pd.to_datetime(test_df['disbursement_date'])
test_df['Year'] = test_df['disbursement_date'].dt.year

print("Train years:", sorted(train_df['Year'].unique()))
print("Test years:", sorted(test_df['Year'].unique()))

In [ ]:
econ_cols = [c for c in econ_pivot.columns if c not in ('country_id', 'Year')]

train_before = train_df.shape[0]
train_df = train_df.merge(econ_pivot, on=['country_id', 'Year'], how='left')
assert train_df.shape[0] == train_before, "Row count changed after merge!"

test_before = test_df.shape[0]
test_df = test_df.merge(econ_pivot, on=['country_id', 'Year'], how='left')
assert test_df.shape[0] == test_before, "Row count changed after merge!"

print("NaN in economic columns -- train:", train_df[econ_cols].isnull().sum().sum())
print("NaN in economic columns -- test:", test_df[econ_cols].isnull().sum().sum())

print("\nTest NaN by country:")
print(test_df.groupby('country_id')[econ_cols].apply(lambda g: g.isnull().sum().sum()))

If the NaN counts above are not 0, stop here and inspect: print `econ_pivot['country_id'].unique()` vs `test_df['country_id'].unique()` to catch any spelling/whitespace mismatch before continuing.

## 6. Feature engineering (NEW)

Cheap, high-signal derived features from columns you already have:
- `repayment_ratio` — implied cost of the loan (how much more you repay vs. borrow)
- `interest_amount` — absolute markup
- `lender_funding_ratio` — how much of the loan the lender actually funded
- `customer_loan_number` — which loan number this is for the customer (1st, 2nd, 3rd...), based on disbursement order. Repeat borrowers often have different risk than first-timers.

These use only information available at (or before) disbursement time, so they shouldn't introduce new leakage.

In [ ]:
for df in (train_df, test_df):
    df['interest_amount'] = df['Total_Amount_to_Repay'] - df['Total_Amount']
    df['repayment_ratio'] = df['Total_Amount_to_Repay'] / df['Total_Amount'].replace(0, np.nan)
    df['repayment_ratio'] = df['repayment_ratio'].fillna(1.0)
    df['lender_funding_ratio'] = df['Amount_Funded_By_Lender'] / df['Total_Amount'].replace(0, np.nan)
    df['lender_funding_ratio'] = df['lender_funding_ratio'].fillna(0.0)

# customer_loan_number: rank of this loan's disbursement date within the customer's history
combined_dates = pd.concat([
    train_df[['customer_id', 'disbursement_date']],
    test_df[['customer_id', 'disbursement_date']]
], ignore_index=True)
combined_dates['customer_loan_number'] = combined_dates.groupby('customer_id')['disbursement_date'].rank(method='first')

train_df['customer_loan_number'] = combined_dates.loc[:len(train_df) - 1, 'customer_loan_number'].values
test_df['customer_loan_number'] = combined_dates.loc[len(train_df):, 'customer_loan_number'].values

print(train_df[['Total_Amount', 'Total_Amount_to_Repay', 'interest_amount',
                 'repayment_ratio', 'lender_funding_ratio', 'customer_loan_number']].head())

## 7. Handle any remaining missing values (non-economic columns)

In [ ]:
missing_train = train_df.columns[train_df.isnull().any()].tolist()
missing_test = test_df.columns[test_df.isnull().any()].tolist()
print("Columns with missing values -- train:", missing_train)
print("Columns with missing values -- test:", missing_test)

numeric_missing = train_df[missing_train].select_dtypes(include='number').columns.tolist() if missing_train else []
categorical_missing = train_df[missing_train].select_dtypes(include='object').columns.tolist() if missing_train else []

for col in numeric_missing:
    med = train_df[col].median()
    med = 0 if pd.isna(med) else med
    train_df[col] = train_df[col].fillna(med)
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(med)

for col in categorical_missing:
    train_df[col] = train_df[col].fillna('missing')
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna('missing')

print("\nRemaining missing -- train:", train_df.isnull().sum().sum())
print("Remaining missing -- test:", test_df.isnull().sum().sum())

## 8. Encode categorical columns

In [ ]:
categorical_cols = train_df.select_dtypes(include='object').columns.tolist()
for c in ['ID']:
    if c in categorical_cols:
        categorical_cols.remove(c)

cardinalities = {c: train_df[c].nunique() for c in categorical_cols}
print("Categorical columns and cardinality:", cardinalities)

label_encode_cols = [c for c, n in cardinalities.items() if n > 15]
one_hot_cols = [c for c, n in cardinalities.items() if n <= 15]
print("Label encode:", label_encode_cols)
print("One-hot encode:", one_hot_cols)

In [ ]:
for col in label_encode_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_df[col] = oe.fit_transform(train_df[[col]])
    test_df[col] = oe.transform(test_df[[col]])

if one_hot_cols:
    train_df = pd.get_dummies(train_df, columns=one_hot_cols)
    test_df = pd.get_dummies(test_df, columns=one_hot_cols)
    train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)
    if 'target' in test_df.columns:
        test_df = test_df.drop(columns=['target'])

print("Final train_df shape:", train_df.shape)
print("Final test_df shape:", test_df.shape)

## 9. Check customer-level leakage risk

In [ ]:
target_variability = train_df.groupby('customer_id')['target'].nunique()
print("Customers with more than one distinct target value:", (target_variability > 1).sum())
print("Total unique customers:", train_df['customer_id'].nunique())
print("Total rows:", train_df.shape[0])
print("--> If total rows >> unique customers, the same customer appears many times, and a naive split leaks.")

## 10. Quick group-based split (by customer_id) — used only to pick LightGBM vs CatBoost

In [ ]:
X = train_df.drop(columns=['target'])
y = train_df['target']
groups = X['customer_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

overlap = set(X_train['customer_id']) & set(X_val['customer_id'])
print("customer_id overlap between train/val:", len(overlap), "(should be 0)")
print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("Target rate -- train:", y_train.mean(), "val:", y_val.mean())

## 11. Train LightGBM

In [ ]:
drop_cols = ['ID', 'disbursement_date', 'tbl_loan_id', 'customer_id']

X_train_model = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val_model = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])

neg, pos = y_train.value_counts()[0], y_train.value_counts()[1]
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

lgb_clf = lgb.LGBMClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
)
lgb_clf.fit(X_train_model, y_train)

lgb_val_proba = lgb_clf.predict_proba(X_val_model)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_val_proba)
print(f"LightGBM validation AUC: {lgb_auc:.4f}")
print("If this is still above ~0.95, there may be another leakage source -- check feature importances below.")

## 12. Train CatBoost (NEW) and compare

In [ ]:
cat_clf = CatBoostClassifier(
    random_state=42,
    iterations=400,
    class_weights=[1, scale_pos_weight],
    verbose=False,
)
cat_clf.fit(X_train_model, y_train)

cat_val_proba = cat_clf.predict_proba(X_val_model)[:, 1]
cat_auc = roc_auc_score(y_val, cat_val_proba)
print(f"CatBoost validation AUC: {cat_auc:.4f}")

print(f"\nLightGBM AUC: {lgb_auc:.4f}")
print(f"CatBoost AUC: {cat_auc:.4f}")

if cat_auc >= lgb_auc:
    best_model_name = "catboost"
    print("\n--> CatBoost selected as best model")
else:
    best_model_name = "lightgbm"
    print("\n--> LightGBM selected as best model")

## 13. Feature importance (sanity check for leakage)

In [ ]:
importances = pd.Series(lgb_clf.feature_importances_, index=X_train_model.columns)
print("LightGBM top features:")
display(importances.sort_values(ascending=False).head(15))

## 14. 5-fold GroupKFold cross-validation (NEW)

Instead of trusting one 80/20 split, this trains 5 models, each on a different 80% of
customers, validating on the other 20%. Every row ends up in exactly one validation fold,
so we get an **out-of-fold (OOF) prediction for the entire training set** -- a much more
robust basis for picking the F1 threshold than a single held-out split.

Test predictions are averaged across all 5 fold-models (a small ensemble), which typically
gives a small but real accuracy boost over any single model.

In [ ]:
N_FOLDS = 5

X_all = train_df.drop(columns=['target'])
y_all = train_df['target']
groups_all = X_all['customer_id']

X_all_model = X_all.drop(columns=[c for c in drop_cols if c in X_all.columns])
X_test_model = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns], errors='ignore')
X_test_model = X_test_model.reindex(columns=X_all_model.columns, fill_value=0)

gkf = GroupKFold(n_splits=N_FOLDS)

oof_proba = np.zeros(len(X_all_model))
test_proba_folds = np.zeros((N_FOLDS, len(X_test_model)))
fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all_model, y_all, groups=groups_all)):
    X_tr, X_va = X_all_model.iloc[tr_idx], X_all_model.iloc[va_idx]
    y_tr, y_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]

    neg_tr, pos_tr = y_tr.value_counts()[0], y_tr.value_counts()[1]
    spw = neg_tr / pos_tr

    if best_model_name == "catboost":
        model = CatBoostClassifier(random_state=42, iterations=400,
                                    class_weights=[1, spw], verbose=False)
    else:
        model = lgb.LGBMClassifier(random_state=42, scale_pos_weight=spw, n_estimators=300)

    model.fit(X_tr, y_tr)

    va_proba = model.predict_proba(X_va)[:, 1]
    oof_proba[va_idx] = va_proba
    fold_auc = roc_auc_score(y_va, va_proba)
    fold_aucs.append(fold_auc)

    test_proba_folds[fold] = model.predict_proba(X_test_model)[:, 1]

    print(f"Fold {fold + 1}/{N_FOLDS} -- AUC: {fold_auc:.4f}")

print(f"\nMean fold AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")
print(f"Overall OOF AUC: {roc_auc_score(y_all, oof_proba):.4f}")

test_proba = test_proba_folds.mean(axis=0)
print("\nAveraged test prediction sample:", test_proba[:5])

## 15. Find best F1 threshold using out-of-fold predictions

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.01)
best_f1, best_th = 0, 0.5
for th in thresholds:
    preds = (oof_proba >= th).astype(int)
    f1 = f1_score(y_all, preds)
    if f1 > best_f1:
        best_f1, best_th = f1, th

print(f"Best F1 (out-of-fold, whole training set): {best_f1:.4f} at threshold {best_th:.2f}")
print("(For comparison, the earlier single-split F1 was based on only 20% of the data -- this number is more trustworthy.)")

## 16. Build submission (ID, target -- binary 0/1)

In [ ]:
test_preds_binary = (test_proba >= best_th).astype(int)

submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'target': test_preds_binary
})

os.makedirs('../outputs', exist_ok=True)
submission_df.to_csv('../outputs/submission_cv.csv', index=False)
print("Saved ../outputs/submission_cv.csv")
print(submission_df['target'].value_counts())
submission_df.head()